# Cosmos Anime rolling training on Colab

Use a GPU runtime. Add `HF_TOKEN` in Colab Secrets after accepting the T5Gemma and dataset terms. The default run uses one 50k-dataset shard as an integration test; increase or remove `SHARD_LIMIT` only after that run succeeds.

In [ ]:
PROJECT_REPO = "https://github.com/YOUR_ACCOUNT/my_sd.git"
PROJECT_BRANCH = "main"
PROJECT_DIR = "/content/my_sd"
DATASET_REPO = "animetimm/danbooru-wdtagger-v4-w640-ws-50k"
DATASET_SPLIT = "train"
SMOKE_RUN = True
SHARD_LIMIT = 1  # Set to None when SMOKE_RUN is False.
FORCE_CONFIG = ""  # Empty selects L4/BF16 or T4/FP16 automatically.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
import subprocess
from pathlib import Path

project = Path(PROJECT_DIR)
if project.joinpath(".git").is_dir():
    subprocess.run(["git", "-C", str(project), "pull", "--ff-only"], check=True)
else:
    if "YOUR_ACCOUNT" in PROJECT_REPO:
        raise ValueError("Set PROJECT_REPO to the Git URL containing this project.")
    subprocess.run(["git", "clone", "--branch", PROJECT_BRANCH, "--depth", "1", PROJECT_REPO, str(project)], check=True)
os.chdir(project)
print(Path.cwd())

In [ ]:
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[train]"], check=True)

In [ ]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Add HF_TOKEN to Colab Secrets and enable notebook access.")
os.environ["HF_TOKEN"] = hf_token
login(token=hf_token, add_to_git_credential=False)

In [ ]:
from huggingface_hub import hf_hub_download

wan_repo = Path("/content/Wan2.2")
if not wan_repo.joinpath("wan/modules/vae2_2.py").is_file():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Wan-Video/Wan2.2.git", str(wan_repo)], check=True)
model_dir = Path("/content/models")
model_dir.mkdir(parents=True, exist_ok=True)
hf_hub_download(repo_id="Wan-AI/Wan2.2-TI2V-5B", filename="Wan2.2_VAE.pth", local_dir=model_dir)
text_dir = model_dir / "t5gemma2-270m-encoder"
if not text_dir.joinpath("config.json").is_file():
    subprocess.run([sys.executable, "scripts/extract_t5gemma_encoder.py", "--output", str(text_dir)], check=True)

In [ ]:
shard_command = [sys.executable, "scripts/list_hf_shards.py", "--repo", DATASET_REPO, "--split", DATASET_SPLIT, "--output", "/content/raw_shards.txt"]
if SHARD_LIMIT is not None:
    shard_command.extend(["--limit", str(SHARD_LIMIT)])
subprocess.run(shard_command, check=True)
print(Path("/content/raw_shards.txt").read_text(encoding="utf-8"))

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before continuing.")
gpu = torch.cuda.get_device_properties(0)
memory_gib = gpu.total_memory / 1024**3
if FORCE_CONFIG:
    config_path = FORCE_CONFIG
elif torch.cuda.is_bf16_supported() and memory_gib >= 20:
    config_path = "configs/colab_l4_rolling.yaml"
else:
    config_path = "configs/colab_t4_rolling.yaml"
print(f"GPU: {gpu.name} ({memory_gib:.1f} GiB); config: {config_path}")

In [ ]:
subprocess.run([sys.executable, "scripts/colab_preflight.py", "--config", config_path], check=True)

In [ ]:
train_command = [sys.executable, "scripts/train.py", "--config", config_path, "--resume", "auto"]
if SMOKE_RUN:
    flavor = "l4" if "l4" in config_path else "t4"
    train_command.extend([
        "--max-steps", "8",
        "--rolling-block-size", "256",
        "--output-dir", f"/content/checkpoints_smoke_{flavor}",
        "--checkpoint-mirror-dir", f"/content/drive/MyDrive/cosmos_anime/checkpoints_smoke_{flavor}",
    ])
elif SHARD_LIMIT is not None:
    raise ValueError("Set SHARD_LIMIT=None for a production run.")
subprocess.run(train_command, check=True)